# Notebook 03 - Tecnicas de Prompting

## Objetivos
- Aplicar zero-shot, one-shot y few-shot prompting.
- Implementar Chain-of-Thought (CoT) para razonamiento.
- Usar role prompting y context prompting con documentos reales.

## Introduccion
No existe una tecnica universal. Cada tarea requiere la estrategia adecuada. En este notebook practicas las 6 tecnicas principales con GPT-2 en español y datasets del curso.

In [1]:
from pathlib import Path
from IPython.display import display, Markdown
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path('..')
DATASETS = BASE / 'datasets'
print('Entorno listo. Datasets:', list(DATASETS.glob('*.csv')))

Entorno listo. Datasets: [WindowsPath('../datasets/casos_practicos.csv'), WindowsPath('../datasets/documentos_empresa.csv'), WindowsPath('../datasets/prompts_buenos_malos.csv'), WindowsPath('../datasets/reviews_sentiment.csv'), WindowsPath('../datasets/tareas_comparacion.csv'), WindowsPath('../datasets/tickets_soporte.csv')]


In [2]:
from transformers import pipeline, set_seed

set_seed(42)
generator = pipeline('text-generation', model='datificate/gpt2-small-spanish')
print('GPT-2 en español listo para experimentos de prompting')

Device set to use cpu


GPT-2 en español listo para experimentos de prompting


In [3]:
def generar(prompt, max_new_tokens=40, temperature=0.7):
    out = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        pad_token_id=generator.tokenizer.eos_token_id,
    )
    return out[0]['generated_text']

print('Funcion generar() lista')

Funcion generar() lista


## 1) Zero-shot: sin ejemplos

In [4]:
def zero_shot(tarea, entrada):
    return f'{tarea}\nTexto: {entrada}\nResultado:'

prompt_zs = zero_shot(
    'Clasifica el sentimiento como positivo, negativo o neutral. Solo la etiqueta.',
    'Excelente producto, llegó rápido y funciona perfectamente.',
)
print('=== ZERO-SHOT ===')
print(generar(prompt_zs, max_new_tokens=10, temperature=0.3))

=== ZERO-SHOT ===
Clasifica el sentimiento como positivo, negativo o neutral. Solo la etiqueta.
Texto: Excelente producto, llegó rápido y funciona perfectamente.
Resultado: Es un producto de alto rendimiento y de alta calidad


## 2) One-shot: un ejemplo

In [5]:
def one_shot(tarea, ejemplo_input, ejemplo_output, entrada):
    return (
        f'{tarea}\n'
        f'Entrada: {ejemplo_input}\n'
        f'Salida: {ejemplo_output}\n'
        f'Entrada: {entrada}\n'
        f'Salida:'
    )

prompt_os = one_shot(
    'Resume en una oracion. Solo el resumen.',
    'Python fue creado por Guido van Rossum en 1991.',
    'Python fue creado en 1991 por Guido van Rossum.',
    'Los Transformers fueron introducidos por Google en 2017.',
)
print('=== ONE-SHOT ===')
print(generar(prompt_os, max_new_tokens=10, temperature=0.3))

=== ONE-SHOT ===
Resume en una oracion. Solo el resumen.
Entrada: Python fue creado por Guido van Rossum en 1991.
Salida: Python fue creado en 1991 por Guido van Rossum.
Entrada: Los Transformers fueron introducidos por Google en 2017.
Salida: Python fue creado por Guido van Rossum en 1992


## 3) Few-shot: multiples ejemplos con tickets

In [6]:
df_tickets = pd.read_csv(DATASETS / 'tickets_soporte.csv')
ejemplos = df_tickets.head(3)
display(ejemplos)

def few_shot_clasificacion(ejemplos_df, nuevo_ticket):
    lineas = ['Clasifica la urgencia del ticket como alta, media o baja. Solo la etiqueta.', '']
    for _, row in ejemplos_df.iterrows():
        lineas.append(f'Ticket: {row["ticket"]}')
        lineas.append(f'Urgencia: {row["urgencia"]}')
        lineas.append('')
    lineas.append(f'Ticket: {nuevo_ticket}')
    lineas.append('Urgencia:')
    return '\n'.join(lineas)

prompt_fs = few_shot_clasificacion(ejemplos, 'El sistema de pagos está caído no puedo procesar reembolsos')
print('=== FEW-SHOT ===')
print(prompt_fs)
print('---')
print(generar(prompt_fs, max_new_tokens=5, temperature=0.3))

,ticket,urgencia,categoria
0,Sistema caido no puedo facturar a ningun cliente,alta,tecnico
1,Como cambio mi contraseña del portal,baja,cuenta
2,Error intermitente en reportes de ventas,media,tecnico


=== FEW-SHOT ===
Clasifica la urgencia del ticket como alta, media o baja. Solo la etiqueta.

Ticket: Sistema caido no puedo facturar a ningun cliente
Urgencia: alta

Ticket: Como cambio mi contraseña del portal
Urgencia: baja

Ticket: Error intermitente en reportes de ventas
Urgencia: media

Ticket: El sistema de pagos está caído no puedo procesar reembolsos
Urgencia:
---
Clasifica la urgencia del ticket como alta, media o baja. Solo la etiqueta.

Ticket: Sistema caido no puedo facturar a ningun cliente
Urgencia: alta

Ticket: Como cambio mi contraseña del portal
Urgencia: baja

Ticket: Error intermitente en reportes de ventas
Urgencia: media

Ticket: El sistema de pagos está caído no puedo procesar reembolsos
Urgencia: baja

Ticket


## 4) Comparacion zero-shot vs modelo de sentimiento en español

In [7]:
sentimientos = pipeline('sentiment-analysis', model='pysentimiento/robertuito-sentiment-analysis')
print('Modelo de sentimiento en español cargado')

Device set to use cpu


Modelo de sentimiento en español cargado


In [8]:
df_reviews = pd.read_csv(DATASETS / 'reviews_sentiment.csv')
resultados = []
for _, row in df_reviews.head(4).iterrows():
    texto = row['texto']
    real = row['sentimiento']
    p_zs = zero_shot('Clasifica sentimiento. Solo etiqueta:', texto)
    r_zs = generar(p_zs, max_new_tokens=5, temperature=0.3)
    r_modelo = sentimientos(texto)[0]
    resultados.append({
        'texto': texto[:45],
        'real': real,
        'zero_shot_gpt': r_zs[-25:],
        'robertuito': r_modelo['label'],
        'score': round(r_modelo['score'], 3),
    })
display(pd.DataFrame(resultados))

,texto,real,zero_shot_gpt,robertuito,score
0,"Excelente producto, llegó rápido y funciona p",positivo,mente.\nResultado:\n\nTexto:,POS,0.976
1,"Muy mala calidad, se rompió al segundo día.",negativo,do: No se puede hacer una,NEG,0.932
2,El servicio al cliente fue amable y resolvió,positivo,l servicio al cliente fue,NEU,0.519
3,Demasiado caro para lo que ofrece.,negativo,"tado: No es una etiqueta,",NEG,0.881


## 5) Chain-of-Thought (CoT)

In [9]:
def chain_of_thought(problema):
    return f'{problema}\nPiensa paso a paso. Muestra el razonamiento y luego la respuesta final.'

problema = 'Una tienda tiene 23 manzanas. Usan 20 para pasteles y compran 6 más. ¿Cuántas manzanas tienen?'
prompt_cot = chain_of_thought(problema)
print('=== CHAIN-OF-THOUGHT ===')
print(generar(prompt_cot, max_new_tokens=60, temperature=0.5))

=== CHAIN-OF-THOUGHT ===
Una tienda tiene 23 manzanas. Usan 20 para pasteles y compran 6 más. ¿Cuántas manzanas tienen?
Piensa paso a paso. Muestra el razonamiento y luego la respuesta final.






























































## 6) Role Prompting

In [10]:
def role_prompt(rol, pregunta):
    return f'Eres {rol}.\nPregunta: {pregunta}\nRespuesta:'

roles = [
    ('un profesor paciente que explica a un niño de 10 años', '¿Qué es una red neuronal?'),
    ('un abogado laboral colombiano', '¿Qué es una cláusula de no competencia?'),
    ('un copywriter creativo de marketing', 'Escribe un eslogan para una marca de café'),
]
for rol, pregunta in roles:
    p = role_prompt(rol, pregunta)
    print(f'\n=== ROLE: {rol[:30]}... ===')
    print(generar(p, max_new_tokens=40, temperature=0.7))


=== ROLE: un profesor paciente que expli... ===
Eres un profesor paciente que explica a un niño de 10 años.
Pregunta: ¿Qué es una red neuronal?
Respuesta: No es la misma red neuronal que la que rodea un objeto o una familia.

Realiza: ¿Cómo se puede describir una red neuronal?
Respuesta: Es una red neuronal que rodea

=== ROLE: un abogado laboral colombiano... ===
Eres un abogado laboral colombiano.
Pregunta: ¿Qué es una cláusula de no competencia?
Respuesta: Si una cláusula de no competencia es el resultado de un problema laboral, entonces no puede existir una cláusula de no competencia.
Respuesta: Si una cláusula de no competencia es la resultado de un problema

=== ROLE: un copywriter creativo de mark... ===
Eres un copywriter creativo de marketing.
Pregunta: Escribe un eslogan para una marca de café
Respuesta: Avota un paquete a la mesa
Repuesta: Un paquete a la mesa.


Grames de edición limitada.

"Grames de edición limitada" es una


## 7) Context Prompting con documentos

In [11]:
df_docs = pd.read_csv(DATASETS / 'documentos_empresa.csv')
doc = df_docs[df_docs['titulo'] == 'Politica de devoluciones'].iloc[0]

def context_prompt(contexto, pregunta):
    return (
        f'Usa SOLO el contexto siguiente. Si no encuentras la respuesta, di No disponible.\n'
        f'<contexto>\n{contexto}\n</contexto>\n'
        f'Pregunta: {pregunta}\n'
        f'Respuesta:'
    )

prompt_ctx = context_prompt(doc['contenido'], '¿Puedo devolver un producto después de 3 semanas?')
print('=== CONTEXT PROMPTING ===')
print(generar(prompt_ctx, max_new_tokens=40, temperature=0.3))

=== CONTEXT PROMPTING ===
Usa SOLO el contexto siguiente. Si no encuentras la respuesta, di No disponible.
<contexto>
Las devoluciones se aceptan dentro de 30 dias calendario. El producto debe estar sin usar y con factura original. No se aceptan productos personalizados. El reembolso se procesa en 5-10 dias habiles.
</contexto>
Pregunta: ¿Puedo devolver un producto después de 3 semanas?
Respuesta: No.








































## Resultados
Practicamos las 6 tecnicas principales con datasets reales: tickets, reseñas y documentos empresariales.

## Conclusiones
Zero-shot para tareas simples; few-shot para formatos propios; CoT para razonamiento; role para expertise; context para QA sobre documentos.

## Ejercicios guiados resueltos
**Ejercicio:** Clasifica un ticket con zero-shot y few-shot y compara.

**Solucion:**

In [12]:
ticket = 'El servidor se cayó y perdí todos mis datos'
zs = generar(zero_shot('Clasifica urgencia: alta/media/baja. Solo etiqueta:', ticket), max_new_tokens=5, temperature=0.3)
fs = generar(few_shot_clasificacion(df_tickets.head(3), ticket), max_new_tokens=5, temperature=0.3)
print('Zero-shot:', zs[-15:])
print('Few-shot:', fs[-15:])

Zero-shot: 
El servidor se
Few-shot: a: baja

Ticket


## Ejercicios propuestos
1. Diseña few-shot con 5 ejemplos propios.
2. Prueba CoT con un problema de tu industria.
3. Context prompting con 2 documentos del CSV.

## Preguntas de reflexion
1. Cuando few-shot empeora en vez de mejorar?
2. CoT siempre mejora la precision?
3. Como elegirias entre role y context prompting?